# ChromBPNet: ATAC-seq Predictions and Variant Analysis

This ChromBPNet pipeline was adapted by Dr Hannah Maude in the Cebola lab. 
The official github for ChromBPNet can be found here https://github.com/kundajelab/chrombpnet?tab=readme-ov-file
The Pre print https://www.biorxiv.org/content/10.1101/2024.12.25.630221v2

## Overview
This notebook guides you through analyzing ATAC-seq chromatin accessibility using ChromBPNet, a deep learning model. It covers:
- Generating SHAP contribution scores for variants of interest
- Calculating ATAC-seq predictions with bias correction
- Visualizing predictions and variant effects
- Generating publication-quality plots

## Prerequisites
Before running this notebook, ensure you have:
1. Run `train_bias_model_sif.sh` to train bias models (5 folds)
2. Run `chrombpnet_pipeline_sif.sh` to generate predictions
3. Run `variant_prediction.sh` to score your variants

## Workflow Steps
1. **Setup**: Define working directory and paths
2. **SHAP Scores**: Calculate variant contribution scores
3. **Sequence Prep**: Create FASTA files for variant regions
4. **Predictions**: Load ChromBPNet model and predict accessibility
5. **Visualization**: Plot predictions and SHAP contributions

## Required Software
- Python 3.9+
- TensorFlow 2.x
- ChromBPNet training library
- Matplotlib/Seaborn for visualization

## Data Requirements
- ChromBPNet trained models (generated by training scripts)
- Reference genome (GRCh38)
- Variant coordinates (in dbSNP or custom format)


In [ ]:
 # IN THE TERMINAL
 # There are two Singularity images available on the general Cebola lab project space, one for ChromBPNet and one for the following Variant-Scorer pipeline.
 # Apptainer is pre-installed on the HPC (CX3 cluster), so you don't need to install anything to run ChromBPNet
 # The Apptainer images are saved in the general lab space
/rds/general/project/cebola_lab_general/live/chrombpnet/chrombpnet.sif
/rds/general/project/cebola_lab_general/live/variant-scorer/variant-scorer.sif

# Step 1a: selecting your threshold


In [ ]:
#Train the enzyme bias model on your ATAC data, this will capture the bias of the Tn5 transposase.
#requires two steps:
#Train one model on different thresholds and select the optimum.
#Then using the optimum threshold, train five different models using different chromosomes for training and validation. 
#To train the bias model, the scripts train_bias_model_sif_fold0.sh and train_bias_model_sif.sh will be used (both saved in /rds/general/project/cebola_lab_general/live/chrombpnet).

# Copy the script for training and selecting your threshold
cp /rds/general/project/cebola_lab_general/live/chrombpnet/train_bias_model_sif_fold0.sh .

#sh train_bias_model_sif.sh --help
#The script usage can be seen by running bash train_bias_model_sif_fold0.sh --help which will show the following:

#Usage: qsub -v BAM=input.bam,PEAKS=peaks.bed,WORKDIR=/path/to/work,THRESHOLD=0.1,NAME=myproject,ASSAY=ATAC train_bias_model_sif_fold0.sh

#Environment variables:
  #BAM      : path to input BAM file (required). Bam file should have duplicates, non-autosomal chromosomes and blacklist regions removed. Index file (.bai) should be present in the same directory, or will be created.
  #PEAKS    : path to input peaks file (required). Peaks should be in narrowPeak format, recommended MACS2 p = 0.01, with blacklist regions removed.
  #WORKDIR  : working directory (optional, default: directory created in the cebola_lab_general ephemeral space)
  #THRESHOLD: optional threshold value (default=0.1)
  #NAME     : name prefix for output files (optional, default="myproject")

#Job uses the following set paths:
 #   genomeDir=/rds/general/project/cebola_lab_general/live/reference-genomes/GRCh38_gencode44/
  #  sifPATH=/rds/general/project/cebola_lab_general/live/chrombpnet
  
  # Recommended code to remove blacklist regions
bedtools slop -i blacklist.bed.gz -g hg38.chrom.sizes -b 1057 > temp.bed
bedtools intersect -v -a overlap.bed.gz -b temp.bed  > peaks_no_blacklist.bed

#Submit the job
# THIS IS AN EXAMPLE
qsub -v BAM=/rds/general/project/cebolalab_liver_regulomes/live/amp_t2d/LSECs/ATAC-seq/pooled_bams/C_ins.filtered.bam,PEAKS=/rds/general/project/cebolalab_liver_regulomes/live/amp_t2d/LSECs/ATAC-seq/ChromBPNet/LSEC_peaks_noBlacklist.narrowPeak,NAME=lsec,THRESHOLD=0.2 train_bias_model_sif_fold0.sh
qsub -v BAM=/rds/general/project/cebolalab_liver_regulomes/live/amp_t2d/LSECs/ATAC-seq/pooled_bams/C_ins.filtered.bam,PEAKS=/rds/general/project/cebolalab_liver_regulomes/live/amp_t2d/LSECs/ATAC-seq/ChromBPNet/LSEC_peaks_noBlacklist.narrowPeak,NAME=lsec,THRESHOLD=0.3 train_bias_model_sif_fold0.sh
qsub -v BAM=/rds/general/project/cebolalab_liver_regulomes/live/amp_t2d/LSECs/ATAC-seq/pooled_bams/C_ins.filtered.bam,PEAKS=/rds/general/project/cebolalab_liver_regulomes/live/amp_t2d/LSECs/ATAC-seq/ChromBPNet/LSEC_peaks_noBlacklist.narrowPeak,NAME=lsec,THRESHOLD=0.4 train_bias_model_sif_fold0.sh
qsub -v BAM=/rds/general/project/cebolalab_liver_regulomes/live/amp_t2d/LSECs/ATAC-seq/pooled_bams/C_ins.filtered.bam,PEAKS=/rds/general/project/cebolalab_liver_regulomes/live/amp_t2d/LSECs/ATAC-seq/ChromBPNet/LSEC_peaks_noBlacklist.narrowPeak,NAME=lsec,THRESHOLD=0.5 train_bias_model_sif_fold0.sh

#Once the job is run, you should select the threshold as appropriate, following the official guidelines for interpreting the QC report.
#The group recommends threshold 0.5 


#The output files are described in detail on the [GitHub](https://github.com/kundajelab/chrombpnet/wiki/Bias-model-training, but should include:

#hg38.chrom.sizes
#output_negatives.bed
#input.bam, input.bam.bai
#peaks.narrowPeak
#output_auxiliary/
#splits/
#output/models/xxxx_xxx_bias.h5 > following your input arguments i.e. ${NAME}_${THRESHOLD}_bias.h5
#output/auxiliary
#output/evaluation - contains all the QC and the overall report.
#output/logs





# Step 1b: training the models with your chosen threshold

In [ ]:
# Copy the script for training five models with your chosen threshold
cp /rds/general/project/cebola_lab_general/live/chrombpnet/train_bias_model_sif.sh .

qsub -v BAM=/rds/general/project/cebolalab_liver_regulomes/live/amp_t2d/LSECs/ATAC-seq/pooled_bams/C_ins.filtered.bam,PEAKS=/rds/general/project/cebolalab_liver_regulomes/live/amp_t2d/LSECs/ATAC-seq/ChromBPNet/LSEC_peaks_noBlacklist.narrowPeak,THRESHOLD=0.2,NAME=lsec train_bias_model_sif.sh

#If you did not specify a working directory (recommended), this will create five new output directories within /rds/general/project/cebola_lab_general/ephemeral/chromBPNet/, in which there should be the output models and reports trained on all five chromosome splits, i.e:

#/rds/general/project/cebola_lab_general/ephemeral/chromBPNet/bias_model_xxxxxx[0].pbs-7
#/rds/general/project/cebola_lab_general/ephemeral/chromBPNet/bias_model_xxxxxx[1].pbs-7
#/rds/general/project/cebola_lab_general/ephemeral/chromBPNet/bias_model_xxxxxx[2].pbs-7
#/rds/general/project/cebola_lab_general/ephemeral/chromBPNet/bias_model_xxxxxx[3].pbs-7
#/rds/general/project/cebola_lab_general/ephemeral/chromBPNet/bias_model_xxxxxx[4].pbs-7

# Step two: run the ChromBPNet pipeline

In [ ]:
cp /rds/general/project/cebola_lab_general/live/chrombpnet/chrombpnet_pipeline_sif.sh .


## The help for running the script is shown below:
#Usage: qsub -v WORKDIR=/path/to/work,MODEL=XXXX_XX_bias.h5,FOLD=fold_0 chrombpnet_pipeline_sif.sh

# Environment variables:
#   WORKDIR  : Working directory. Use folder created by train_bias_model_sif.sh if run. Must contain peaks.narrowPeak, input.bam and input.bam.bai. (required).
#   MODEL    : Name of bias model file. Should be a file name within output_fold_x/models/, e.g. XXXX_XX_bias.h5 (required).
#   FOLD     : Specify training fold (fold_x). Results folder will be called results_x. Default is fold_0.

# Job uses the following set paths:
#   genomeDir=/rds/general/project/cebola_lab_general/live/reference-genomes/GRCh38_gencode44/
#  sifPATH=/rds/general/project/cebola_lab_general/live/chrombpnet

#For best practise, use a WORKDIR in the ephemeral project space and copy results back to permanent live storage after job completion.
#ESSENTIAL: the results folder must NOT already exist in WORKDIR otherwise the job will exit. For failed attempts, delete the existing results folder before re-running.

###
#You'll need to train five ChromBPNet models, using the five bias models. 
# Read the offical GitHub for descriptions of the output files and how to interpret the overall report (found in the results/evaluation folder).

# The XXXX_XX_bias.h5 files will follow the format ${NAME}_{THRESHOLD} 
# Replace the bias_model_xxxxxx with the name for your folders
# The 1-5 here are trained the provided splits which are numbered 0-4.
qsub -v WORKDIR=/rds/general/project/cebola_lab_general/ephemeral/chromBPNet/bias_model_xxxxxx[1].pbs-7,MODEL=XXXX_XX_bias.h5,FOLD=fold_0 chrombpnet_pipeline_sif.sh
qsub -v WORKDIR=/rds/general/project/cebola_lab_general/ephemeral/chromBPNet/bias_model_xxxxxx[2].pbs-7,MODEL=XXXX_XX_bias.h5,FOLD=fold_1 chrombpnet_pipeline_sif.sh
qsub -v WORKDIR=/rds/general/project/cebola_lab_general/ephemeral/chromBPNet/bias_model_xxxxxx[3].pbs-7,MODEL=XXXX_XX_bias.h5,FOLD=fold_2 chrombpnet_pipeline_sif.sh
qsub -v WORKDIR=/rds/general/project/cebola_lab_general/ephemeral/chromBPNet/bias_model_xxxxxx[4].pbs-7,MODEL=XXXX_XX_bias.h5,FOLD=fold_3 chrombpnet_pipeline_sif.sh
qsub -v WORKDIR=/rds/general/project/cebola_lab_general/ephemeral/chromBPNet/bias_model_xxxxxx[5].pbs-7,MODEL=XXXX_XX_bias.h5,FOLD=fold_4 chrombpnet_pipeline_sif.sh

# Exploring individual variants 
### Bigwig visualisation

In [ ]:
#ChromBPNet predicts traces for 1kb blocks and can generate bigwigs using the chromBPNet pred_bw command. 
# It requires a 10-column bed file as input. I have prepared a script called makeBed_for_pred_bw.sh, which will generate the required bed file surrounding your query variant.

# # # Run everything in the box below on the terminal from a Jupyter server with GPU. 
# https://jupyter.cx3.rcs.ic.ac.uk/hub/login?next=%2Fhub%2F select 4 cores, 32GB, 8 hours, 1 Nividia

# EDIT THESE: run the lines on the terminal to set the variable to values specific to your project
# Match the NAME and THRESHOLD values used previously, as the command will look for the model files with these names
RANGE="100kb"; ID="rs72840103"; THRESHOLD="0.2"; POS="99922157"; CHR="chr10"; NAME="lsec"

# Set your working directory
WORKDIR=/rds/general/project/cebolalab_liver_regulomes/live/amp_t2d/LSECs/ATAC-seq/ChromBPNet/chromBPNet_0.2
# The code assumes that WORKDIR contains the files results_fold_x/models/chrombpnet.h5 and results_fold_x/models/chrombpnet_nobias.h5


# Change to the working directory
cd "${WORKDIR}"
mkdir variant-investigations

# Make the bed file
PATH=$PATH:/rds/general/project/cebola_lab_general/live/chrombpnet/
makeBed_for_pred_bw.sh --chr "${CHR}" --pos "${POS}" --range "${RANGE}" --id "${ID}"

# View the bed file
head "${ID}"_"${RANGE}".bed

mkdir variant-investigations/"${ID}"
mv "${ID}"_"${RANGE}".bed variant-investigations/"${ID}"

# Set paths - don't edit these, just paste the lines to the terminal
genomeDir=/rds/general/project/cebola_lab_general/live/reference-genomes/GRCh38_gencode44/
sifPATH=/rds/general/project/cebola_lab_general/live/chrombpnet

# --- Run for all five models folds ---
for i in {0..4}
do
apptainer exec --nv \
    --bind "${genomeDir}":/genome \
    --bind "${WORKDIR}":/work \
    --env PYTHONPATH=/scratch/variant-scorer/src \
    "${sifPATH}"/chrombpnet.sif \
    chrombpnet pred_bw \
    -bm /work/output_fold_"${i}"/models/"${NAME}"_"${THRESHOLD}"_bias.h5 \
    -cm /work/results_fold_"${i}"/models/chrombpnet.h5 \
    -cmb /work/results_fold_"${i}"/models/chrombpnet_nobias.h5 \
    -r /work/variant-investigations/"${ID}"/"${ID}"_"${RANGE}".bed \
    -g /genome/GCA_000001405.15_GRCh38_no_alt_analysis_set.fna \
    -c /work/hg38.chrom.sizes \
    -op /work/variant-investigations/"${ID}"/"${NAME}"_"${ID}"_"${RANGE}"_fold_"${i}"
done


# Make an environment with your preferred conda management tool - e.g. conda, micromamba, miniforge
# activate conda, then activate micromamba

eval "$(~/anaconda3/bin/conda shell.bash hook)"
eval "$(micromamba shell hook --shell bash)"
# Set up the environment
micromamba create -n wiggletools -c bioconda wiggletools ucsc-bedgraphtobigwig ucsc-wigtobigwig
# Activate the environment
micromamba activate wiggletools

#Three types of bigwigs are produced: _bias.bw, _chrombpnet.bw, _chrombpnet_nobias.bw.
#Generate bigwigs based on the mean of the five training folds as follows:

# Edit lsec to the name you have used previously
NAME="lsec_${ID}_${RANGE}"
# Calculate the mean of the bigwigs across the five training folds
wiggletools write "${NAME}"_mean_chrombpnet_nobias.wig mean "${NAME}"_fold_0_chrombpnet_nobias.bw "${NAME}"_fold_1_chrombpnet_nobias.bw "${NAME}"_fold_2_chrombpnet_nobias.bw "${NAME}"_fold_3_chrombpnet_nobias.bw "${NAME}"_fold_4_chrombpnet_nobias.bw
wiggletools write "${NAME}"_mean_chrombpnet.wig mean "${NAME}"_fold_0_chrombpnet.bw "${NAME}"_fold_1_chrombpnet.bw "${NAME}"_fold_2_chrombpnet.bw "${NAME}"_fold_3_chrombpnet.bw "${NAME}"_fold_4_chrombpnet.bw
wiggletools write "${NAME}"_mean_bias.wig mean "${NAME}"_fold_0_bias.bw "${NAME}"_fold_1_bias.bw "${NAME}"_fold_2_bias.bw "${NAME}"_fold_3_bias.bw "${NAME}"_fold_4_bias.bw

# Convert wig to BigWig using hg38.chrom.sizes
wigToBigWig "${NAME}"_mean_chrombpnet_nobias.wig "${WORKDIR}"/hg38.chrom.sizes "${NAME}"_mean_chrombpnet_nobias.bigwig
wigToBigWig "${NAME}"_mean_chrombpnet.wig "${WORKDIR}"/hg38.chrom.sizes "${NAME}"_mean_chrombpnet.bigwig
wigToBigWig "${NAME}"_mean_bias.wig "${WORKDIR}"/hg38.chrom.sizes "${NAME}"_mean_bias.bigwig

# Predict ATAC-seq from a custom sequence and SHAP contribution scores

Run everything in the box below on the terminal from a Jupyter server with GPU. 
https://jupyter.cx3.rcs.ic.ac.uk/hub/login?next=%2Fhub%2F select 4 cores, 32GB, 8 hours, 1 Nividia

In [ ]:
# IN THE TERMINAL of Jupyter
# Set your working directory as WORKDIR and set your variant identifier
WORKDIR=/rds/general/user/tv722/projects/tamara_vujic_phd/live/ChromBPNet/chrombpnet0.5
rsid="rs986009256"
# Change directory to WORKDIR
cd "${WORKDIR}"
# Set paths
genomeDir=/rds/general/project/cebola_lab_general/live/reference-genomes/GRCh38_gencode44/
sifPATH=/rds/general/project/cebola_lab_general/live/variant-scorer/

# Create a directory called variant-investigations plus a subdirectory for your variant (if they don't exist)
mkdir -p variant-investigations/"${rsid}"

Calculate the SHAP contribution scores:

First, you should run the variant_shap.py script to calculate the SHAP contribution scores for every base around your variant(s) of interest. The input format for your variant should be a file called variant-investigations/"${rsid}"/"${rsid}".tsv in the following format:

In [ ]:
# IN THE TERMINAL of Jupyter
# Generate the SHAP contribution scores for all five folds
# Should take less than 10 minutes for one variant
for i in {0..4}
do
apptainer exec --nv \
--bind "${genomeDir}":/genome \
--bind "${WORKDIR}":/work \
--env PYTHONPATH=/scratch/variant-scorer/src \
"${sifPATH}"/variant-scorer.sif \
python /scratch/variant-scorer/src/variant_shap.py \
--list /work/variant-investigations/"${rsid}"/"${rsid}".tsv \
-g /genome/GCA_000001405.15_GRCh38_no_alt_analysis_set.fna \
-s /work/hg38.chrom.sizes \
-m /work/results_fold_"${i}"/models/chrombpnet_nobias.h5 \
-o /work/variant-investigations/"${rsid}"/"${rsid}"_fold_"${i}"+shap \
-sc chrombpnet \
--shap_type counts profile
done

Create your fasta file:

You need to create a fasta file(s) containing your input sequence. If you have a variant of interest, the quickest way is to to the the UCSC Genome Browser (GRCh38/hg38) > View > DNA Sequence. Input your variant position and extend the sequence +/-1,057bp.
Save the fasta file as variant-investigations/{rsid}/{rsid}.fa.

Plot ATAC-seq predictions and SHAP contributions:

Now, open Python on the terminal by typing in the following:

In [ ]:
# IN THE TERMINAL of Jupyter
# Set these in your shell first (adjust only if paths change)
WORKDIR="/rds/general/user/tv722/projects/tamara_vujic_phd/live/ChromBPNet/chrombpnet0.5"
genomeDir="/rds/general/project/cebola_lab_general/live/reference-genomes/GRCh38_gencode44"
sifPATH="/rds/general/project/cebola_lab_general/live/chrombpnet"

# Start Python inside the container, in /work
apptainer exec --nv \
--bind "${genomeDir}":/genome \
--bind "${WORKDIR}":/work \
--pwd /work \
--env PYTHONPATH=/scratch/variant-scorer/src \
"${sifPATH}/chrombpnet.sif" \
python

In [ ]:
# OPEN Python in Jupyter
# --- Load required package ---
import tensorflow as tf
from tensorflow.keras.models import load_model
import chrombpnet.training.utils.losses as losses
import chrombpnet.training.utils.one_hot as one_hot
from tensorflow.keras.utils import get_custom_objects
from tensorflow.keras.models import load_model
import numpy as np

import matplotlib
matplotlib.use("PDF") # or "SVG". Avoids error due to missing matplotlib.backends.backend_agg
import matplotlib.pyplot as plt

# -- EDIT VARAIBLES HERE --    
ref="G"
alt="T"
rsid="rs986009256"
# --- END EDIT VARIABLES ---

In [ ]:
# -- Define functions ---
# IMPORTANT: paste each function in individually and press enter until you see the prompt return
def load_sequences_from_fasta(fasta_path):
    sequences = []
    with open(fasta_path) as f:
        current_seq = []
        for line in f:
            if line.startswith(">"):
                if current_seq:
                    sequences.append("".join(current_seq))
                    current_seq = []
            else:
                current_seq.append(line.strip())
                
        if current_seq:
            assert len("".join(current_seq)) == 2114, "Input sequence must be exactly 2114 bp long."
            sequences.append("".join(current_seq))
    return sequences

In [ ]:
# load_model_wrapper function
def load_model_wrapper(model_h5):
    # read .h5 model
    custom_objects={"multinomial_nll":losses.multinomial_nll, "tf": tf}    
    get_custom_objects().update(custom_objects)    
    model=load_model(model_h5, compile=False)
    #model.summary()
    return model
    

In [ ]:
# softmax function
def softmax(x, temp=1):
        norm_x = x - np.mean(x,axis=1, keepdims=True)
        return np.exp(temp*norm_x)/np.sum(np.exp(temp*norm_x), axis=1, keepdims=True)

In [ ]:
# ---- Step 1: Input your custom 2114bp sequence. 
# Make sure it only contains A, C, G, T
sequences = load_sequences_from_fasta(f"variant-investigations/{rsid}/{rsid}.fa")

# One-hot encode the sequence
one_hot_seqs = one_hot.dna_to_one_hot(sequences)  # shape: (N, 2114, 4)
print("One-hot shape:", one_hot_seqs.shape)

# ---- Step 2: Load models & get predictions ----
# Set the five model paths
model_paths = [
    "results_fold_0/models/chrombpnet_nobias.h5",
    "results_fold_1/models/chrombpnet_nobias.h5",
    "results_fold_2/models/chrombpnet_nobias.h5",
    "results_fold_3/models/chrombpnet_nobias.h5",
    "results_fold_4/models/chrombpnet_nobias.h5",
]

# ---- use the model here ----
# This code loads each of the five model folds one by one to save memory
# Predictions for the input sequence will be added to the predictions dictionary
predictions = {} # Creates a dictionary to store the results

for i, model_path in enumerate(model_paths):
    print(f"Loading model {i}")
    model = load_model_wrapper(model_path)
    pred_logits_wo_bias, pred_logcts_wo_bias = model.predict(one_hot_seqs)
    print("Prediction logits shape:", pred_logits_wo_bias.shape) # (N, 1000)
    print("Prediction logcnt shape:", pred_logcts_wo_bias.shape) # (N, 1)
    preds = (softmax(pred_logits_wo_bias) * np.exp(pred_logcts_wo_bias[:, 0])[:, None])
    predictions[i] = preds
    # cleanup
    del pred_logits_wo_bias, pred_logcts_wo_bias, preds, model
    tf.keras.backend.clear_session()
# Press enter to complete

In [ ]:
print("Prediction shape:", predictions[i].shape) # (N, 1000)
# Now predictions dict contains predictions from all five folds

# Calculate the average across folds
avg_predictions = np.mean(
    np.stack([predictions[i] for i in sorted(predictions.keys())], axis=0),
    axis=0
).astype(np.float32)

In [ ]:
# ---- Step 4: Plot a specific window around the variant ----
# Assume predictions.shape = (N, 1000)
pred_len = avg_predictions.shape[1]  # 1000
mid_idx = pred_len // 2             # 500

# Edit window size as preferred
window_size = 1000 # any value <=1000
half_window = window_size // 2
start_idx = mid_idx - half_window
end_idx = mid_idx + half_window

# Slice predictions
pred_window = avg_predictions[:, start_idx:end_idx]

# x-axis: prediction indices
x_coords = np.arange(start_idx, end_idx)

# Plot index 0 and index 1
plt.figure(figsize=(10,4))

plt.plot(x_coords, pred_window[0], label=ref+" allele",linewidth=2, color='#1A85FF')
plt.plot(x_coords, pred_window[1], label=alt+" allele",linewidth=2, color='#D41159')

plt.xlabel("Prediction index")
plt.ylabel("Prediction")
plt.title(str(window_size)+"-window around "+rsid)
plt.legend()
plt.savefig(f"variant-investigations/{rsid}/{rsid}_{str(window_size)}_pred.pdf", bbox_inches="tight")  # tight trims excess white space
plt.show()
plt.close()

In [ ]:
# Continue in Python
# Import additional packages
import h5py
import seaborn as sns
import os 
import hdf5plugin
import logomaker
import pandas as pd
from scipy.ndimage import gaussian_filter1d
# Packages already loaded above
#import numpy as np
#import matplotlib.pyplot as plt

# Set the paths to the h5 files
# Set file paths
import os, glob


counts_paths = [
    f"variant-investigations/{rsid}/{rsid}_fold_0+shap.variant_shap.counts.h5",
    f"variant-investigations/{rsid}/{rsid}_fold_1+shap.variant_shap.counts.h5",
    f"variant-investigations/{rsid}/{rsid}_fold_2+shap.variant_shap.counts.h5",
    f"variant-investigations/{rsid}/{rsid}_fold_3+shap.variant_shap.counts.h5",
    f"variant-investigations/{rsid}/{rsid}_fold_4+shap.variant_shap.counts.h5",
]

print("counts_paths type:", type(counts_paths))
for p in counts_paths:
    print(p, "-> isfile:", os.path.isfile(p))

import os, h5py

profile_paths = [
    f"variant-investigations/{rsid}/{rsid}_fold_0+shap.variant_shap.profile.h5",
    f"variant-investigations/{rsid}/{rsid}_fold_1+shap.variant_shap.profile.h5",
    f"variant-investigations/{rsid}/{rsid}_fold_2+shap.variant_shap.profile.h5",
    f"variant-investigations/{rsid}/{rsid}_fold_3+shap.variant_shap.profile.h5",
    f"variant-investigations/{rsid}/{rsid}_fold_4+shap.variant_shap.profile.h5",
]

# Verify files exist
for p in profile_paths:
    print(p, "isfile:", os.path.isfile(p))

with h5py.File(counts_paths[0], "r") as f:
    print("Rows in h5:", f["alleles"].shape[0])
    print("Unique alleles:", set(f["alleles"][:]))
    print("Variant IDs:", len(f["variant_ids"]))

In [ ]:
# --- Counts ---
# Creat empty dictionaries to hold data
import h5py

alleles = {}
raw_seq = {}
shap_seq = {}
proj_shap_seq = {}
variant_ids = {}

for i, h5file in enumerate(counts_paths):
    print(f"Loading fold {i} from {h5file}")
    with h5py.File(h5file, "r") as f:
        alleles[i] = f["alleles"][:]                  # shape (2*num_variants,)
        raw_seq[i] = f["raw/seq"][:]                  # one-hot sequences
        shap_seq[i] = f["shap/seq"][:]                # SHAP values
        proj_shap_seq[i] = f["projected_shap/seq"][:] # multiplied by raw
        variant_ids[i] = f["variant_ids"][:]


# --- Profiles ---
alleles_p = {}
raw_seq_p = {}
shap_seq_p = {}
proj_shap_seq_p = {}
variant_ids_p = {}

for i, h5file in enumerate(profile_paths):
    print(f"Loading profile fold {i} from {h5file}")
    with h5py.File(h5file, "r") as f:
        alleles_p[i] = f["alleles"][:]
        raw_seq_p[i] = f["raw/seq"][:]
        shap_seq_p[i] = f["shap/seq"][:]
        proj_shap_seq_p[i] = f["projected_shap/seq"][:]
        variant_ids_p[i] = f["variant_ids"][:]

In [ ]:
# Calculate the average across folds
proj_shap_seq_avg = np.mean(
    np.stack([proj_shap_seq[i] for i in sorted(proj_shap_seq.keys())], axis=0),
    axis=0
).astype(np.float32)

# Calculate the average across folds
proj_shap_seq_avg_p = np.mean(
    np.stack([proj_shap_seq_p[i] for i in sorted(proj_shap_seq_p.keys())], axis=0),
    axis=0
).astype(np.float32)

In [ ]:
variant_ids_str=['G','T']
def plot_shap_contributions(idx, proj_shap=proj_shap_seq_avg, window_size=1000, lowlim=-0.1, uplim=0.13, rsid="rsID"):
	"""Plot SHAP contributions for a given allele."""
	# Plot window
	start = 1057 - (window_size // 2)
	end = 1057 + (window_size // 2)
	# Get the projected shap sequence for the query region
	allele = proj_shap[idx][:, start:end]  # shape (4, seq_len)
	seq_len = allele.shape[1]
	positions = np.arange(seq_len)
	# Sum contributions across nucleotides (A/C/G/T)
	total_contrib = allele.sum(axis=0)#.astype(np.float32)  # shape (seq_len,)
	#smoothed = gaussian_filter1d(total_contrib, sigma=1)  # sigma controls smoothness
	# Plot
	plt.figure(figsize=(12, 3))	
	plt.ylim(lowlim, uplim)
	# Variant position
	variant_pos = seq_len // 2
	# Fill positive and negative contributions
	plt.fill_between(range(seq_len), 
		     np.where(allele.sum(axis=0) > 0, allele.sum(axis=0), 0), 
		     color='maroon', alpha=0.6, label='Positive contributions')
	plt.fill_between(range(seq_len), 
		     np.where(allele.sum(axis=0) < 0, allele.sum(axis=0), 0), 
		     color='darkblue', alpha=0.6, label='Negative contributions')
	# Add vertical line for variant position
	plt.axvline(x=variant_pos, color='black', linestyle='--', label='Variant position', linewidth=0.2, alpha=1)
	plt.xlabel('Sequence Position')
	plt.ylabel('SHAP Contribution')
	plt.title('SHAP Contributions Across Sequence')
	plt.legend()
	allele = variant_ids_str[idx]
	plt.savefig(f"variant-investigations/{rsid}/SHAP_contribution_{rsid}_{allele}_{window_size}bp_window.pdf")
	plt.show()

In [ ]:
# Plot contributions to the predicted ATAC-seq counts
plot_shap_contributions(idx=0, proj_shap=proj_shap_seq_avg, window_size=1000,lowlim=-0.02, uplim=0.05, rsid=rsid)
plot_shap_contributions(idx=1, proj_shap=proj_shap_seq_avg, window_size=1000,lowlim=-0.02, uplim=0.05, rsid=rsid)

# Plot contributions to the predicted ATAC-seq profile
plot_shap_contributions(idx=0, proj_shap=proj_shap_seq_avg_p, window_size=1000,lowlim=-0.02, uplim=0.05, rsid=rsid)
plot_shap_contributions(idx=1, proj_shap=proj_shap_seq_avg_p, window_size=1000,lowlim=-0.02, uplim=0.05, rsid=rsid)

In [ ]:
variant_ids_str=['G','T']
def plot_shap_logo(idx, proj_shap=proj_shap_seq_avg, mode="counts", window_size=1000, lowlim=-0.1, uplim=0.13, rsid="rsID"):
	"""Plot SHAP contribution logo for a given allele."""
	# Plot window
	start = 1057 - (window_size // 2)
	end = 1057 + (window_size // 2)
	allele = variant_ids_str[idx]
	# Full projected SHAP array (4, 2114)
	allele_full = proj_shap[idx]
	# Slice the region (still shape (4, window_len))
	allele_zoom = allele_full[:, start:end]
	# Convert to DataFrame for logomaker
	df = pd.DataFrame(allele_zoom.T, columns=["A", "C", "G", "T"])
	# Plot
	plt.figure(figsize=(12, 3))
	logomaker.Logo(df, shade_below=0, fade_below=0.1, color_scheme="classic")
	# Add vertical line at position 100
	#plt.axvline(x=100, color="red", linestyle="--", linewidth=1)
	plt.title(f"Projected SHAP logo for {rsid} {allele} allele, nPositions {start}-{end}")
	plt.xlabel("Position (relative)")
	plt.ylim(lowlim, uplim)  # or any range you want
	plt.savefig(f"variant-investigations/{rsid}/SHAP_logo_{rsid}_{allele}_{window_size}bp_window_{mode}.pdf")
	plt.show()

In [ ]:
# Edit the low and up limits as preferred
# Edit the low and up limits as preferred
plot_shap_logo(idx=0, proj_shap=proj_shap_seq_avg, mode="counts", window_size=20, lowlim=-0.02, uplim=0.03, rsid=rsid)
plot_shap_logo(idx=1, proj_shap=proj_shap_seq_avg_p, mode="profile", window_size=20, lowlim=-0.02, uplim=0.03, rsid=rsid)

# Variant effect predictor
## Variant scoring


This analysis applies the ChromBPNet models trained above to "score" the impact of a variant on chromatin accessibility
The model takes the sequence surrounding a variant and compares the predicted ATAC-seq trace for the sequence containing the reference allele vs the alternative allele
The variant scoring pipeline from Kundajelab is hard coded to expect 10,000 variants as input
Your input variant file MUST match the reference genome you use as input, otherwise the reference alleles may not match and the pipeline will throw an error.
The variant scoring only works for SNPs

How to quickly obtain 10,000 variants: My advice is to download common variants from the UCSC Table Browser. Use the following parameters:

Mammal > Human > GRCh38
Group: Variation > dbSnp155
Table: Common dbSNP (155)
Region of interest: set to your chromosome of interest, e.g. chr21
Output format: all fields
Output filename: set a name, download the file and upload it to the HPC (e.g. "chr21_dbSnp155Common").
You want to end up with exactly 10,000 SNVs (indels won't work). The format for running variant-scorer should be:

['chr', 'pos', 'allele1', 'allele2', 'variant_id']

The coordinates need to be expanded by ±1,057 so don't include any variants at the end of chromosomes where that addition/subtraction will take the coordinates to below 0 or above the length of the chromosome.


In [ ]:
# Edit the 10000 if you plan to manually add variants 
# E.g. if adding 2 variants, use 9998
awk -v FS='\t' -v OFS='\t' '{split($7,a,","); if (length($5)==1 && length(a[1])==1) print $1,$3,$5,a[1],$4}' chr21_dbSnp155Common | sed '1d' | head -n 10000 >  chr21_dbSnp155Common.chrombpnet.tsv

#PBS -l walltime=08:00:00
#PBS -l select=1:ncpus=4:mem=64gb:ngpus=1:gpu_type=L40S
#PBS -N variant-scorer
#PBS -J 0-4

tid=$PBS_ARRAY_INDEX

# --- Edit to your WORKDIR ---
WORKDIR=/rds/general/project/...
NAME="xxxxx"

# --- Set paths ---
genomeDir=/rds/general/project/cebola_lab_general/live/reference-genomes/GRCh38_gencode44/
sifPATH=/rds/general/project/cebola_lab_general/live/variant-scorer

# Run the variant-scorer pipeline
# variant_scoring.py
apptainer exec --nv \
	--bind "${genomeDir}":/genome \
	--bind "${WORKDIR}":/work \
	--env PYTHONPATH=/scratch/variant-scorer/src \
	"${sifPATH}"/variant-scorer.sif \
	python /scratch/variant-scorer/src/variant_scoring.py \
	-l /work/"${NAME}".tsv \
	-g /genome/GCA_000001405.15_GRCh38_no_alt_analysis_set.fna \
	-s /work/hg38.chrom.sizes \
	-m /work/results_fold_"${tid}"/models/chrombpnet_nobias.h5 \
	-o /work/results_fold_"${tid}"/"${NAME}"_variant_scores \
	-sc chrombpnet \
	-dm

In [ ]:
### You then need to average the scores across the five folds, which you can do in R. Example code below.

# Run in R
suppressPackageStartupMessages(library(ggplot2))
suppressPackageStartupMessages(library(tidyverse))
suppressPackageStartupMessages(library(ggrepel))

# Set working directory and read results
setwd('/rds/general/project/...')

# Create lists to store data
scores=list()
# Read file and add results dataframe for each fold to the promoters, enhancers and other lists
for(fold in 0:4){
  name <- paste0("fold_",fold)
  scores[[name]] <- read.table(paste0('fold_', fold, '_variant_scores.tsv'), header=TRUE, sep='\t')
}

rsIDs <- scores[[1]]$variant_id
means = matrix(nrow=length(rsIDs), ncol=7, dimnames=list(rsIDs, c("chr", "pos", "ref", "alt", "rsID", "mean_logfc", "mean_pval"))) %>% as.data.frame()
means[,1:5] <- scores[[1]][,1:5]

# For all variants, calculate the mean logfc and mean p-value
for(id in rsIDs){
	means[id, "mean_logfc"] <- lapply(scores, function(x) {x[x$variant_id == id, ]$logfc}) %>% unlist() %>% as.numeric %>% mean()
	means[id, "mean_pval"] <- lapply(scores, function(x) {x[x$variant_id == id, ]$logfc.pval}) %>% unlist() %>% as.numeric %>% mean() 
	means[id, "allele1_pred_counts"] <- lapply(scores, function(x) {x[x$variant_id == id, ]$allele1_pred_counts}) %>% unlist() %>% as.numeric %>% mean()
	means[id, "allele2_pred_counts"] <- lapply(scores, function(x) {x[x$variant_id == id, ]$allele2_pred_counts}) %>% unlist() %>% as.numeric %>% mean()
}

# Save the output file
write.table(means, file = "mean_variant_scores.tsv", sep='\t', quote=FALSE)